# LOKLOK Thesis Final Upgrade — Full Run (T4 GPU)

This notebook runs the final calibrated in-silico experiment, regenerates the figures, and packages the outputs. It is deterministic given the seed.

**Runtime:** Google Colab, T4 GPU (or CPU fallback).
**Expected time:** 1–3 minutes for the full 500-persona, 18-session run.
**Outputs:** `03_data/model_outputs/`, `04_figures/`, and `loklok_outputs.zip`.

## 1. Check the runtime

In [ ]:
!nvidia-smi || echo 'No GPU attached — the pipeline will use the NumPy CPU fallback.'

## 2. Install dependencies

In [ ]:
!pip -q install numpy scipy pandas matplotlib scikit-learn networkx nbformat
import numpy, scipy, pandas, matplotlib, sklearn, networkx
print('numpy', numpy.__version__)
print('scipy', scipy.__version__)
print('pandas', pandas.__version__)
print('matplotlib', matplotlib.__version__)

## 3. Upload or mount the package

Option A: upload `colab_bundle.zip` from the package root. Option B: mount Google Drive and point `ROOT` at the synced folder.

In [ ]:
from google.colab import files
import os, zipfile, pathlib
ROOT = pathlib.Path('/content/loklok-thesis-final-upgrade')
if not ROOT.exists():
    print('Please upload colab_bundle.zip (or skip and set ROOT to your Drive folder).')
    up = files.upload()
    for name in up:
        with zipfile.ZipFile(name, 'r') as z:
            z.extractall('/content')
print('ROOT exists:', ROOT.exists())
print(sorted(p.name for p in ROOT.iterdir())[:10] if ROOT.exists() else 'missing')

## 4. Run the final pipeline

In [ ]:
import subprocess, sys
cmd = [sys.executable, str(ROOT / '02_code' / 'pipeline' / 'loklok_pipeline.py'),
       '--root', str(ROOT), '--n-personas', '500', '--sessions', '18', '--seed', '20261105']
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 5. Regenerate the publication figures

In [ ]:
subprocess.run([sys.executable, str(ROOT / '02_code' / 'pipeline' / 'generate_figures.py'),
                '--root', str(ROOT)], check=True)

## 6. Inspect key outputs

In [ ]:
import json, pandas as pd
res = json.loads((ROOT / '03_data' / 'model_outputs' / 'pipeline_results.json').read_text())
fa = res['factorial']
print('Delta SReval:', round(fa['delta_sr_eval']['mean'], 3), fa['delta_sr_eval']['ci_low'], fa['delta_sr_eval']['ci_high'])
print('Delta SRcomp:', round(fa['delta_sr_comp']['mean'], 2))
print('Delta engagement:', round(fa['delta_engagement']['mean'], 3))
print('Sobol survival:', res['sensitivity']['survival_rate_pct'], '%')
pd.DataFrame(fa['cell_means']).head()

## 7. Display the main figures

In [ ]:
from IPython.display import Image, display
for name in ['fig6_1_sr_distributions.png', 'fig6_2_factorial_cells.png',
             'fig6_5_pareto_hamiltonian.png', 'fig6_7_forest.png',
             'fig6_9_misinfo.png', 'fig6_10_sobol.png']:
    p = ROOT / '04_figures' / 'png' / name
    if p.exists():
        display(Image(filename=str(p), width=850))

## 8. Package and download the outputs

In [ ]:
import shutil
zip_path = '/content/loklok_outputs'
shutil.make_archive(zip_path, 'zip', root_dir=str(ROOT),
                    base_dir='03_data/model_outputs')
# include figures and generated tables too
shutil.make_archive('/content/loklok_figures', 'zip', root_dir=str(ROOT), base_dir='04_figures')
from google.colab import files
files.download('/content/loklok_outputs.zip')
files.download('/content/loklok_figures.zip')

### Notes

- The seed is fixed at **20261105**; rerunning should reproduce the same numbers.
- The pipeline prints a hardware line (`PyTorch_CUDA` or `NumPy_CPU_Fallback`).
- The Bayesian prior-sensitivity result is part of the output; do not select only the favourable prior.
- All results are **model-based**. The measured N=52 wave is an anchor, not a treatment arm.